## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

## 2. Carga de datos

### 2.1 Cruce de amparos

In [ ]:
detalle_amparos = pd.read_csv('data/cruce_amparos.csv', sep=';')


In [ ]:
detalle_amparos

,AMPARO,AMPARO DETALLE,Amparo_Id,Apertura_Amparo_Desc
0,Auxilio de repatriación,AUXILIO DE REPATRIACIÓN,55431,RESTO
1,Auxilio funerario,BONO O AUXILIO FUNERARIO,417,RESTO
2,Auxilio funerario,GASTOS EXEQUIALES,817,RESTO
3,Auxilio funerario,Auxilio funerario por cuotas,60081,RESTO
4,Auxilio por maternidad/paternidad,AUXILIO POR MATERNIDAD O PATERNIDAD,55348,RESTO
5,Auxilio por maternidad/paternidad,Auxilio por maternidad o paternidad,63712,RESTO
6,Bono canasta,BONO CANASTA,59976,RESTO
7,Bono canasta,BONO FAMILIAR,17712,RESTO
8,Bono canasta,Bono Funerario,63711,RESTO
9,Bono canasta,Bono canasta,63714,RESTO


### 2.2 Asegurados vigentes (PES y VidaIntegral)

In [ ]:
archivos_asegurados = {
    'PES': 'data/Detalle_Asegurados_Vigentes_PES_202512.txt',
    'VidaIntegral': 'data/Detalle_Asegurados_Vigentes_VidaIntegral_202512.txt',
}

frames_asegurados = []
for producto, ruta in archivos_asegurados.items():
    df_tmp = pd.read_csv(ruta, sep='|', encoding='latin-1', low_memory=False)
    df_tmp['PRODUCTO_ORIGEN'] = producto
    frames_asegurados.append(df_tmp)

bd_asegurados = pd.concat(frames_asegurados, ignore_index=True)
print('Registros por producto:')
print(bd_asegurados['PRODUCTO_ORIGEN'].value_counts())
bd_asegurados.head()

Registros por producto:
PRODUCTO_ORIGEN
VidaIntegral    4185230
PES             1533374
Name: count, dtype: int64


,Amparo_Id,Mes_Id,Poliza_Id,Numero_Poliza,Nombre_Comercial,Producto_Desc,Codigo_Ramo_Op,Nombre_Canal_Comercial,Nombre_Grupo_Canal_Comercial,Poliza_Certificado_Id,...,VALOR_ASEGURADO_VIDA,VALOR_ASEGURADO_TODAS_COBERTURAS,Pct_Tasa_Prima,Factor_Tasa,Tasa_Actual_Peso,Prima_Actual_Vigencia,Prima_Actual_Vigencia_Vida,Limpieza,Factor_Tasa_Limpieza,PRODUCTO_ORIGEN
0,417,202512,244124500,83004322802,PLAN EMPRESARIO SURA CONTRIBUT,PRODUCTO VIDA GRUPO ESTÃNDAR CONTRIBUTIVO,83,SUCURSALES,ASESORES,244325748,...,0,2000000,"2,852",1000.0,"0,002852",5.704,0,0.0,1000.0,PES
1,417,202512,244124500,83004322802,PLAN EMPRESARIO SURA CONTRIBUT,PRODUCTO VIDA GRUPO ESTÃNDAR CONTRIBUTIVO,83,SUCURSALES,ASESORES,273013427,...,0,2000000,"2,852",1000.0,"0,002852",5.704,0,0.0,1000.0,PES
2,417,202512,244124500,83004322802,PLAN EMPRESARIO SURA CONTRIBUT,PRODUCTO VIDA GRUPO ESTÃNDAR CONTRIBUTIVO,83,SUCURSALES,ASESORES,275198183,...,0,2000000,"2,852",1000.0,"0,002852",5.704,0,0.0,1000.0,PES
3,417,202512,244124500,83004322802,PLAN EMPRESARIO SURA CONTRIBUT,PRODUCTO VIDA GRUPO ESTÃNDAR CONTRIBUTIVO,83,SUCURSALES,ASESORES,271232503,...,0,2000000,"2,852",1000.0,"0,002852",5.704,0,0.0,1000.0,PES
4,417,202512,244124500,83004322802,PLAN EMPRESARIO SURA CONTRIBUT,PRODUCTO VIDA GRUPO ESTÃNDAR CONTRIBUTIVO,83,SUCURSALES,ASESORES,270244666,...,0,2000000,"2,852",1000.0,"0,002852",5.704,0,0.0,1000.0,PES


In [ ]:
pd.crosstab(bd_asegurados['PRODUCTO_ORIGEN'], bd_asegurados['Nombre_Comercial'])

Nombre_Comercial,AP INTEGRACIÃN NO CONTRIBUTIVO,PLAN EMPRESARIO SURA CONTRIBUT,PLAN EMPRESARIO SURA NO CONTRI,PLAN VIDA CLÃSICO CONTRIBUTIVO,PLAN VIDA CLÃSICO NO CONTRIBUT,PLAN VIDA DEUDORES,PLAN VIDA INTEGRAL,PLAN VIDA INTEGRAL CONTRIBUTIV,PLAN VIDA INTEGRAL NO CONTRIBU,PLAN VIDA SURA INTEGRAL - 084,PLAN VIDA SURA INTEGRAL - 183,PLAN VIDA SURA INTEGRAL CONTRI,PLAN VIDA SURA INTEGRAL NO CON,SURENTA INTEGRACIÃN NO CONTRI,Vida Grupo IntegraciÃ³n Contrib
PRODUCTO_ORIGEN,,,,,,,,,,,,,,,
PES,0,1453204,80170,0,0,0,0,0,0,0,0,0,0,0,0
VidaIntegral,327,0,0,55959,55336,5,40958,1638992,1431041,9639,7753,599409,343946,370,1495


### 2.3 Pólizas y curvas de riesgo

> `bd_poliza`: info comercial por póliza (canal, grupo, TPR NMAC, etc.)  
> `bd_curvas`: tasas puras de riesgo por edad y amparo


In [ ]:
bd_curvas = pd.read_csv('data/curvas_riesgo.csv', sep=';')
bd_curvas = pd.melt(bd_curvas, id_vars=['EDAD_CLIENTE'], var_name='Amparo_Id', value_name='TPR')
bd_curvas['Amparo_Id'] = pd.to_numeric(bd_curvas['Amparo_Id'], errors='coerce')

## 3. Limpieza y preparación de `bd_asegurados`

### 3.1 Conversión de tipos numéricos

Los campos `VALOR_ASEGURADO` y `VALOR_PRIMA` vienen como texto con coma decimal.

In [ ]:
for col in ['VALOR_ASEGURADO', 'VALOR_PRIMA']:
    bd_asegurados[col] = bd_asegurados[col].str.replace(',', '.', regex=False)
    bd_asegurados[col] = pd.to_numeric(bd_asegurados[col], errors='coerce')


### 3.2 Merge con detalle de amparos

In [ ]:
bd_asegurados = pd.merge(bd_asegurados, detalle_amparos, how='left', on='Amparo_Id')
bd_asegurados.sort_values(
    by=['Asegurado_Id', 'Numero_Poliza', 'Poliza_Certificado_Id', 'VALOR_ASEGURADO'],
    ascending=True,
    inplace=True
)


### 3.3 Consolidación de registros duplicados

**Paso 1**: Misma `Poliza_Certificado_Id` + mismo amparo → tomar VA máximo  
*(ocurre cuando hay duplicados del mismo certificado con distinto VA)*

In [ ]:
bd_asegurados = bd_asegurados.groupby(
    ['Asegurado_Id', 'Numero_Poliza', 'Poliza_Certificado_Id', 'AMPARO']
).agg({
    'Amparo_Id':                   'min',
    'Mes_Id':                      'first',
    'Poliza_Id':                   'first',
    'Nombre_Comercial':            'first',
    'Nombre_Grupo_Canal_Comercial':'first',
    'Edad_Cliente':                'first',
    'Sexo_Cd':                     'first',
    'VALOR_ASEGURADO':             'max',
    'VALOR_PRIMA':                 'sum'
}).reset_index()


**Paso 2**: Diferente `Poliza_Certificado_Id` + mismo amparo → sumar VA  
*(ocurre cuando hay aumento de VA con extraprima, registrado como nuevo certificado)*

In [ ]:
bd_asegurados = bd_asegurados.groupby(
    ['Asegurado_Id', 'Numero_Poliza', 'AMPARO']
).agg({
    'Amparo_Id':                   'min',
    'Mes_Id':                      'first',
    'Poliza_Id':                   'first',
    'Nombre_Comercial':            'first',
    'Nombre_Grupo_Canal_Comercial':'first',
    'Edad_Cliente':                'first',
    'Sexo_Cd':                     'first',
    'VALOR_ASEGURADO':             'sum',
    'VALOR_PRIMA':                 'sum'
}).reset_index()


### 3.4 Ajuste VA Enfermedades Graves

Para EG Independiente el siniestro se paga al 60% del VA de Vida.

In [ ]:
bd_asegurados.loc[
    bd_asegurados['AMPARO'] == 'EG INDEPENDIENTE', 'VALOR_ASEGURADO'
] *= 0.6


### 3.5 Merge con curvas de riesgo

In [ ]:
bd_asegurados = bd_asegurados.merge(
    bd_curvas,
    left_on=['Amparo_Id', 'Edad_Cliente'],
    right_on=['Amparo_Id', 'EDAD_CLIENTE'],
    how='left',
    indicator=True
)

print('Resultado del merge con curvas:')
print(bd_asegurados['_merge'].value_counts())
print(bd_asegurados['VALOR_ASEGURADO'].dtype)
print(bd_asegurados['VALOR_ASEGURADO'].head(5).tolist())

C:\Users\fredaran\AppData\Local\Temp\ipykernel_28288\13470799.py:1: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  bd_asegurados = bd_asegurados.merge(


Resultado del merge con curvas:
_merge
both          4976314
left_only       71602
right_only          0
Name: count, dtype: int64
float64
[4500000.0, 20498400.0, 34164000.0, 34164000.0, 15000000.0]


### 3.6 Filtro de registros inválidos

Se remueven registros con edad = -1, VA nulo o VA ≤ 3 (valores sin sentido técnico).  
**Decisión**: estos representan ~3.6% del total → se descartan.

In [ ]:
# Asegurar conversión numérica antes de cualquier comparación
bd_asegurados['VALOR_ASEGURADO'] = pd.to_numeric(
    bd_asegurados['VALOR_ASEGURADO'].astype(str).str.replace(',', '.', regex=False),
    errors='coerce'
)
bd_asegurados['Edad_Cliente'] = pd.to_numeric(bd_asegurados['Edad_Cliente'], errors='coerce')
bd_asegurados['Amparo_Id']    = pd.to_numeric(bd_asegurados['Amparo_Id'],    errors='coerce')

total = len(bd_asegurados)
print(f"Total registros: {total:,}\n")

# ── Columnas con nulos / vacíos / -1 ──────────────────────────────────────────
cols_check = ['Asegurado_Id', 'Numero_Poliza', 'Amparo_Id']

print("=== Checks por columna (nulos / vacíos / -1) ===")
for col in cols_check:
    nulos     = bd_asegurados[col].isna().sum()
    vacios    = (bd_asegurados[col].astype(str).str.strip() == '').sum()
    menos_uno = (pd.to_numeric(bd_asegurados[col], errors='coerce') == -1).sum()
    problema  = nulos + vacios + menos_uno
    print(f"\n{col}")
    print(f"  Nulos/NaN  : {nulos:>7,}  ({nulos/total*100:.2f}%)")
    print(f"  Vacíos ''  : {vacios:>7,}  ({vacios/total*100:.2f}%)")
    print(f"  Igual a -1 : {menos_uno:>7,}  ({menos_uno/total*100:.2f}%)")
    print(f"  Subtotal   : {problema:>7,}  ({problema/total*100:.2f}%)")

# ── Edad_Cliente: no nula, != -1, != 0 ────────────────────────────────────────
edad_nulos = bd_asegurados['Edad_Cliente'].isna().sum()
edad_cero  = (bd_asegurados['Edad_Cliente'] == 0).sum()
edad_men1  = (bd_asegurados['Edad_Cliente'] == -1).sum()
edad_prob  = edad_nulos + edad_cero + edad_men1
print(f"\nEdad_Cliente")
print(f"  Nulos/NaN  : {edad_nulos:>7,}  ({edad_nulos/total*100:.2f}%)")
print(f"  Igual a 0  : {edad_cero:>7,}  ({edad_cero/total*100:.2f}%)")
print(f"  Igual a -1 : {edad_men1:>7,}  ({edad_men1/total*100:.2f}%)")
print(f"  Subtotal   : {edad_prob:>7,}  ({edad_prob/total*100:.2f}%)")

# ── VALOR_ASEGURADO: no nulo, > 3 ─────────────────────────────────────────────
va_nulos = bd_asegurados['VALOR_ASEGURADO'].isna().sum()
va_men3  = (bd_asegurados['VALOR_ASEGURADO'] <= 3).sum()
va_prob  = va_nulos + va_men3
print(f"\nVALOR_ASEGURADO")
print(f"  Nulos/NaN  : {va_nulos:>7,}  ({va_nulos/total*100:.2f}%)")
print(f"  <= 3       : {va_men3:>7,}  ({va_men3/total*100:.2f}%)")
print(f"  Subtotal   : {va_prob:>7,}  ({va_prob/total*100:.2f}%)")

# ── Resumen combinado ──────────────────────────────────────────────────────────
mask_problema = (
    bd_asegurados[cols_check].isna().any(axis=1) |
    bd_asegurados[cols_check].astype(str).apply(lambda c: c.str.strip() == '').any(axis=1) |
    (bd_asegurados[cols_check].apply(pd.to_numeric, errors='coerce') == -1).any(axis=1) |
    bd_asegurados['Edad_Cliente'].isna() |
    bd_asegurados['Edad_Cliente'].isin([-1, 0]) |
    bd_asegurados['VALOR_ASEGURADO'].isna() |
    (bd_asegurados['VALOR_ASEGURADO'] <= 3)
)
filas_prob = mask_problema.sum()
print("\n" + "="*50)
print(f"Filas con al menos 1 problema : {filas_prob:>7,}")
print(f"Sobre el total                : {filas_prob/total*100:.2f}%")
print(f"Filas que quedarían           : {total - filas_prob:>7,}")
print(f"Retenidas                     : {(total-filas_prob)/total*100:.2f}%")

Total registros: 5,047,916

=== Checks por columna (nulos / vacíos / -1) ===

Asegurado_Id
  Nulos/NaN  :       0  (0.00%)
  Vacíos ''  :       0  (0.00%)
  Igual a -1 :   6,505  (0.13%)
  Subtotal   :   6,505  (0.13%)

Numero_Poliza
  Nulos/NaN  :       0  (0.00%)
  Vacíos ''  :       0  (0.00%)
  Igual a -1 :       0  (0.00%)
  Subtotal   :       0  (0.00%)

Amparo_Id
  Nulos/NaN  :       0  (0.00%)
  Vacíos ''  :       0  (0.00%)
  Igual a -1 :       0  (0.00%)
  Subtotal   :       0  (0.00%)

Edad_Cliente
  Nulos/NaN  :       0  (0.00%)
  Igual a 0  :     418  (0.01%)
  Igual a -1 :  71,602  (1.42%)
  Subtotal   :  72,020  (1.43%)

VALOR_ASEGURADO
  Nulos/NaN  :       0  (0.00%)
  <= 3       :  79,280  (1.57%)
  Subtotal   :  79,280  (1.57%)

Filas con al menos 1 problema : 149,569
Sobre el total                : 2.96%
Filas que quedarían           : 4,898,347
Retenidas                     : 97.04%


In [ ]:
bd_asegurados = bd_asegurados[~mask_problema].copy().reset_index(drop=True)

print(f"Registros antes  : {total:,}")
print(f"Registros después: {len(bd_asegurados):,}")
print(f"Removidos        : {total - len(bd_asegurados):,} ({(total - len(bd_asegurados))/total*100:.2f}%)")

Registros antes  : 5,047,916
Registros después: 4,898,347
Removidos        : 149,569 (2.96%)


## 4. Construcción de variables por tipo de amparo

Se identifican los amparos de cada tipo por su `Amparo_Id`:
- **Vida**: 930, 63717
- **EG**: 221
- **ITP**: 366, 404, 63721, 63722, 63723

### 4.1 Valores asegurados por cobertura

In [ ]:
AMPAROS_VIDA = [930, 63717]
AMPAROS_EG   = [221]
AMPAROS_ITP  = [366, 404, 63721, 63722, 63723]
AMPAROS_VIDA_EG_ITP = AMPAROS_VIDA + AMPAROS_EG + AMPAROS_ITP

bd_asegurados['VA_VIDA'] = 0.0
bd_asegurados.loc[bd_asegurados['Amparo_Id'].isin(AMPAROS_VIDA), 'VA_VIDA'] = \
    bd_asegurados.loc[bd_asegurados['Amparo_Id'].isin(AMPAROS_VIDA), 'VALOR_ASEGURADO']

bd_asegurados['VA_EG'] = 0.0
bd_asegurados.loc[bd_asegurados['Amparo_Id'].isin(AMPAROS_EG), 'VA_EG'] = \
    bd_asegurados.loc[bd_asegurados['Amparo_Id'].isin(AMPAROS_EG), 'VALOR_ASEGURADO']

bd_asegurados['VA_ITP'] = 0.0
bd_asegurados.loc[bd_asegurados['Amparo_Id'].isin(AMPAROS_ITP), 'VA_ITP'] = \
    bd_asegurados.loc[bd_asegurados['Amparo_Id'].isin(AMPAROS_ITP), 'VALOR_ASEGURADO']


### 4.2 Siniestralidad esperada por cobertura

In [ ]:
bd_asegurados['Sin_esperada'] = bd_asegurados['VALOR_ASEGURADO'] * bd_asegurados['TPR']

bd_asegurados['Sin_esp_Vida'] = 0.0
bd_asegurados.loc[bd_asegurados['Amparo_Id'].isin(AMPAROS_VIDA), 'Sin_esp_Vida'] = \
    bd_asegurados.loc[bd_asegurados['Amparo_Id'].isin(AMPAROS_VIDA), 'Sin_esperada']

bd_asegurados['Sin_esp_Vida_EG_ITP'] = 0.0
bd_asegurados.loc[bd_asegurados['Amparo_Id'].isin(AMPAROS_VIDA_EG_ITP), 'Sin_esp_Vida_EG_ITP'] = \
    bd_asegurados.loc[bd_asegurados['Amparo_Id'].isin(AMPAROS_VIDA_EG_ITP), 'Sin_esperada']


## 5. Agregación a nivel póliza (`bd_pol`)

### 5.1 Groupby por `Numero_Poliza`

In [ ]:
bd_pol = bd_asegurados.groupby('Numero_Poliza').agg(
    VALOR_ASEGURADO        = ('VALOR_ASEGURADO',      'sum'),
    VA_VIDA                = ('VA_VIDA',               'sum'),
    Sin_esperada           = ('Sin_esperada',          'sum'),
    Sin_esp_Vida           = ('Sin_esp_Vida',          'sum'),
    Sin_esp_Vida_EG_ITP    = ('Sin_esp_Vida_EG_ITP',  'sum'),
    EDAD_CLIENTE           = ('EDAD_CLIENTE',          'mean'),
    Asegurado_Id           = ('Asegurado_Id',          'nunique')
).reset_index()


### 5.2 Tasas puras de riesgo (TPR) a nivel póliza

Se calculan como siniestralidad esperada / VA_VIDA × 1000 (por mil).

In [ ]:
bd_pol['TPR_ponderada_por_persona'] = 1000 * bd_pol['Sin_esperada']        / bd_pol['VA_VIDA']
bd_pol['TPR_solo_vida']             = 1000 * bd_pol['Sin_esp_Vida']         / bd_pol['VA_VIDA']
bd_pol['TPR_solo_vida_EG_ITP']      = 1000 * bd_pol['Sin_esp_Vida_EG_ITP'] / bd_pol['VA_VIDA']


### 5.3 Merge con `bd_poliza` (info comercial)

Se usa `outer` para detectar pólizas que estén en uno solo de los dos lados.

### 5.4 Conversión de `TPR_NMAC` y cálculo de diferencias

In [ ]:
if 'TPR_NMAC' in bd_pol.columns:
    bd_pol['TPR_NMAC'] = pd.to_numeric(bd_pol['TPR_NMAC'], errors='coerce')
    bd_pol['Diferencias'] = bd_pol['TPR_ponderada_por_persona'] - bd_pol['TPR_NMAC']
else:
    bd_pol['TPR_NMAC'] = np.nan
    bd_pol['Diferencias'] = np.nan
    print("Aviso: 'TPR_NMAC' no existe en bd_pol. Se deja como NaN.")

Aviso: 'TPR_NMAC' no existe en bd_pol. Se deja como NaN.


## 6. Limpieza de `bd_pol`

### 6.1 Checks de calidad

In [ ]:
cols_check = ['EDAD_CLIENTE', 'Asegurado_Id']
total = len(bd_pol)
print(f'Total filas: {total}\n')
print('=== Checks por columna (nulos / -1) ===')

for col in cols_check:
    nulos     = bd_pol[col].isna().sum()
    vacios    = (bd_pol[col].astype(str).str.strip() == '').sum()
    menos_uno = (bd_pol[col] == -1).sum()
    problema  = nulos + vacios + menos_uno
    print(f'\n{col}')
    print(f'  Nulos/NaN  : {nulos:>6}  ({nulos/total*100:.2f}%)')
    print(f'  Vacíos \'\'  : {vacios:>6}  ({vacios/total*100:.2f}%)')
    print(f'  Igual a -1 : {menos_uno:>6}  ({menos_uno/total*100:.2f}%)')
    print(f'  Subtotal   : {problema:>6}  ({problema/total*100:.2f}%)')

va_nulos  = bd_pol['VALOR_ASEGURADO'].isna().sum()
va_men1   = (bd_pol['VALOR_ASEGURADO'] == -1).sum()
va_men3   = (bd_pol['VALOR_ASEGURADO'] <= 3).sum()
print(f'\nVALOR_ASEGURADO')
print(f'  Nulos/NaN  : {va_nulos:>6}  ({va_nulos/total*100:.2f}%)')
print(f'  Igual a -1 : {va_men1:>6}  ({va_men1/total*100:.2f}%)')
print(f'  <= 3       : {va_men3:>6}  ({va_men3/total*100:.2f}%)')

mask_problema = (
    bd_pol[cols_check].isna().any(axis=1) |
    bd_pol[cols_check].astype(str).apply(lambda c: c.str.strip() == '').any(axis=1) |
    (bd_pol[cols_check] == -1).any(axis=1) |
    bd_pol['VALOR_ASEGURADO'].isna() |
    (bd_pol['VALOR_ASEGURADO'] <= 3)
)
filas_prob = mask_problema.sum()
print('\n' + '='*45)
print(f'Filas con al menos 1 problema : {filas_prob:>6}')
print(f'Sobre el total                : {filas_prob/total*100:.2f}%')
print(f'Filas que quedarían           : {total - filas_prob:>6}')
print(f'Retenidas                     : {(total-filas_prob)/total*100:.2f}%')


Total filas: 16215

=== Checks por columna (nulos / -1) ===

EDAD_CLIENTE
  Nulos/NaN  :      0  (0.00%)
  Vacíos ''  :      0  (0.00%)
  Igual a -1 :      0  (0.00%)
  Subtotal   :      0  (0.00%)

Asegurado_Id
  Nulos/NaN  :      0  (0.00%)
  Vacíos ''  :      0  (0.00%)
  Igual a -1 :      0  (0.00%)
  Subtotal   :      0  (0.00%)

VALOR_ASEGURADO
  Nulos/NaN  :      0  (0.00%)
  Igual a -1 :      0  (0.00%)
  <= 3       :      0  (0.00%)

Filas con al menos 1 problema :      0
Sobre el total                : 0.00%
Filas que quedarían           :  16215
Retenidas                     : 100.00%


### 6.2 Remoción de filas problemáticas

In [ ]:
bd_pol = bd_pol[~mask_problema].reset_index(drop=True)

print(f'Filas antes  : {total:,}')
print(f'Filas después: {len(bd_pol):,}')
print(f'Removidas    : {total - len(bd_pol):,}')


Filas antes  : 16,215
Filas después: 16,215
Removidas    : 0


### 6.3 `bd_pol` final y exportacion para visualizacion

## 7. Conversión de tipos antes de exportación

In [ ]:
# Conversión de tipos: valores asegurados a INT, tasas a float
# Esto asegura que VALOR_ASEGURADO salga sin decimales (.0) y tas ratios con precisión controlada

# Convertir valores asegurados a INT (sin decimales)
bd_pol['VALOR_ASEGURADO'] = bd_pol['VALOR_ASEGURADO'].fillna(0).astype('int64')
bd_pol['VA_VIDA']          = bd_pol['VA_VIDA'].fillna(0).astype('int64')

# Mantener siniestralidad esperada como float (para decimales)
bd_pol['Sin_esperada']           = bd_pol['Sin_esperada'].astype('float64')
bd_pol['Sin_esp_Vida']           = bd_pol['Sin_esp_Vida'].astype('float64')
bd_pol['Sin_esp_Vida_EG_ITP']    = bd_pol['Sin_esp_Vida_EG_ITP'].astype('float64')

# Mantener tasas como float
bd_pol['TPR_ponderada_por_persona'] = bd_pol['TPR_ponderada_por_persona'].astype('float64')
bd_pol['TPR_solo_vida']             = bd_pol['TPR_solo_vida'].astype('float64')
bd_pol['TPR_solo_vida_EG_ITP']      = bd_pol['TPR_solo_vida_EG_ITP'].astype('float64')

# Edad como float  
bd_pol['EDAD_CLIENTE']     = bd_pol['EDAD_CLIENTE'].astype('float64')

# Asegurado_Id como int
bd_pol['Asegurado_Id']     = bd_pol['Asegurado_Id'].fillna(0).astype('int64')

print("Conversión de tipos completada:")
print(f"\nTipos de datos después de convertir:")
print(bd_pol.dtypes)
print(f"\nMuestra de datos (primeras 3 filas):")
print(bd_pol.head(3))

In [ ]:
output_dir = Path('output')
output_dir.mkdir(parents=True, exist_ok=True)

ruta_csv = output_dir / 'bd_pol_procesado.csv'
ruta_xlsx = output_dir / 'bd_pol_procesado.xlsx'

# Exportar a CSV con formato colombiano (coma decimal para floats)
bd_pol.to_csv(ruta_csv, index=False, encoding='utf-8-sig', decimal=',')

# Exportar a Excel sin cambios (Excel adaptará automáticamente según el locale)
bd_pol.to_excel(ruta_xlsx, index=False)

print(f'Shape final: {bd_pol.shape}')
print(f'\nColumnas: {list(bd_pol.columns)}')
print(f'\nTipos de datos:')
for col in bd_pol.columns:
    print(f'  {col}: {bd_pol[col].dtype}')
print(f'\nArchivo CSV generado: {ruta_csv}')
print(f'Archivo Excel generado: {ruta_xlsx}')
print(f'\n✓ CSV exportado con decimales separados por COMA (formato colombiano)')
print(f'✓ VALOR_ASEGURADO: tipo INT64 (sin .0)')
print(f'✓ Tasas y siniestralidad: tipo FLOAT64 (con precisión)')

Shape final: (16215, 13)

Columnas: ['Numero_Poliza', 'VALOR_ASEGURADO', 'VA_VIDA', 'Sin_esperada', 'Sin_esp_Vida', 'Sin_esp_Vida_EG_ITP', 'EDAD_CLIENTE', 'Asegurado_Id', 'TPR_ponderada_por_persona', 'TPR_solo_vida', 'TPR_solo_vida_EG_ITP', 'TPR_NMAC', 'Diferencias']

Archivo CSV generado: output\bd_pol_procesado.csv
Archivo Excel generado: output\bd_pol_procesado.xlsx
